# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR\^2 Croissant Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We'll walk through metadata exploration, record set access, basic visualization, and common processing steps.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

All dataset components (record sets, fields, columns) are referenced by their Croissant `@id`.

In [ ]:
# Install mlcroissant if not already available
!pip install --quiet mlcroissant

## 1. Data Loading

Load the dataset metadata and records with `mlcroissant`. This ensures we interact with the latest records and schema from the FAIR^2 dataset. 

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview

Review available record sets, fields, and their Croissant `@id`. This gives you a map of the relational data structure for querying with mlcroissant.

In [ ]:
# List available record sets and details
print("Record Sets:")
for rs in metadata.recordSets:
    print(f"@id: {rs.id}, name: {rs.name}, description: {rs.description}")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, name: {f.name}, dataType: {f.dataType}")

## 3. Data Extraction

Load data from a specific record set (referenced by its `@id`) into a pandas DataFrame for interactive exploration. Use field `@id`s for column access.

*The record set IDs and their field IDs can be found in the previous overview section.*

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in metadata.recordSets]

# Load data for all record sets into DataFrames using @id as key
dfs = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)

# Show available columns for the first record set as an example
if record_set_ids:
    print("Columns in first record set:")
    print(dfs[record_set_ids[0]].columns.tolist())
    display(dfs[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

We'll process one record set (the main protein set, assuming the most detailed record set) by selecting a numeric field (e.g., `cr:coverage` or `cr:peptideCount`) using its `@id`, filtering by value, normalizing, and grouping by a categorical field (such as `cr:description`). Adjust the field IDs as per your actual overview results.

In [ ]:
# Choose primary record set and field IDs (update as appropriate after reviewing Section 2's output)
# Example for illustration only: set these to real @ids for your dataset as needed
main_record_set_id = record_set_ids[0] if record_set_ids else None
# Example numeric field: 'cr:coverage', categorical/grouping: 'cr:description'
# Replace these field IDs with the correct ones from your dataset
numeric_field_id = None
group_field_id = None
if main_record_set_id:
    fields = dfs[main_record_set_id].columns.tolist()
    print("Fields:", fields)

    # Try to pick a numeric field (e.g. coverage or peptideCount), group field (e.g. description)
    # Here we attempt to auto-pick by crude heuristics
    fnum = [f for f in fields if ('coverage' in f or 'peptide' in f or 'MW' in f or 'abundance' in f) and f != '']
    fcat = [f for f in fields if ('desc' in f or 'name' in f or 'accession' in f)]
    if fnum: numeric_field_id = fnum[0]
    if fcat: group_field_id = fcat[0]
    print(f"Numeric field candidate: {numeric_field_id}")
    print(f"Group field candidate: {group_field_id}")

    threshold = dfs[main_record_set_id][numeric_field_id].mean() if numeric_field_id else 0
    if numeric_field_id:
        # Filter on the numeric field (e.g. top values only)
        filtered_df = dfs[main_record_set_id][dfs[main_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (first 5):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field_id} (first 5):")
            display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and any grouped aggregation using `matplotlib` or `seaborn` as appropriate. This aids in spotting trends, outliers, and grouping effects in protein data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and numeric_field_id in dfs[main_record_set_id].columns:
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(dfs[main_record_set_id][numeric_field_id], bins=30, ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    sns.boxplot(y=dfs[main_record_set_id][numeric_field_id], ax=axes[1])
    axes[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
if group_field_id and numeric_field_id and 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion

In this notebook, we've shown how to:
- Load and review a Croissant dataset and metadata with `mlcroissant`
- Explore available record sets and their field structures via `@id`
- Extract all records using record set `@id` and load them into pandas DataFrames
- Perform basic EDA: filtering, normalization, and grouping on selected fields
- Visualize numeric field distributions and grouped summaries

Use the discovered field and record set `@id`s to further analyze and visualize data according to your research interests. For further detail, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and dataset schema.